# Subtaller 3: Introducción a Apache Spark — Arquitectura Medallion Local

## Objetivo
Aprender los fundamentos de Apache Spark mediante la implementación de una arquitectura **Medallion** (Bronze → Silver → Gold) de forma local, usando un dataset de ventas de e-commerce.

## Arquitectura Medallion
```
JSON crudo                 Bronze               Silver                  Gold
(data/)          →   (datos sin limpiar)  →  (datos limpios)  →  (datos agregados)
ventas_ecommerce.json   Parquet raw         Parquet limpio         Parquet analítico
```

## Contenido
1. Instalación y verificación de PySpark
2. Conceptos clave de Spark
3. Crear SparkSession
4. Capa Bronze — Ingestar datos crudos
5. Capa Silver — Limpiar y transformar
6. Capa Gold — Agregar para análisis
7. Consultas analíticas sobre Gold
8. Instructivo: Migrar a AWS S3 + Glue

---
## 1. Instalación y verificación

In [1]:
# Instalar PySpark si no está instalado
# !pip install pyspark

import pyspark
print(f'PySpark version: {pyspark.__version__}')

PySpark version: 4.1.1


---
## 2. Conceptos clave de Spark

| Concepto | Descripción |
|---|---|
| **SparkSession** | Punto de entrada a Spark. Una por aplicación. |
| **DataFrame** | Tabla distribuida con columnas tipadas. Equivale a un DataFrame de Pandas pero distribuido. |
| **RDD** | Resilient Distributed Dataset — capa de bajo nivel (hoy se prefiere DataFrame). |
| **Transformación** | Operación lazy: `filter`, `select`, `groupBy`. No ejecuta hasta una acción. |
| **Acción** | Dispara la ejecución: `show()`, `count()`, `write`, `collect()`. |
| **Partición** | Chunk del dataset distribuido entre nodos (en local, entre CPU cores). |
| **Lazy Evaluation** | Spark construye un plan de ejecución y lo optimiza antes de correrlo. |
| **Catalyst Optimizer** | Motor interno que optimiza el plan de consulta automáticamente. |

---
## 3. Crear SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

# --- INYECCIÓN DE ENTORNO JAVA PARA MAC ---
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
# ----------------------------------------

# SparkSession es el punto de entrada de cualquier aplicación Spark
spark = SparkSession.builder \
    .appName('TallerMedallion') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .getOrCreate()

# Reducir logs para mayor legibilidad
spark.sparkContext.setLogLevel('WARN')

print(f'Spark version: {spark.version}')
print(f'Master: {spark.sparkContext.master}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/03 22:02:40 WARN Utils: Your hostname, MacBook-Air-de-Samuel.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.70 instead (on interface en0)
26/04/03 22:02:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 22:02:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Master: local[*]


In [4]:
# Definir rutas de la arquitectura Medallion
BASE_PATH = './medallion'
BRONZE_PATH = f'{BASE_PATH}/bronze/ventas'
SILVER_PATH = f'{BASE_PATH}/silver/ventas'
GOLD_PATH   = f'{BASE_PATH}/gold'

# Ruta del dataset fuente
SOURCE_JSON = './data/ventas_ecommerce.json'

# Crear directorios locales
for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    os.makedirs(path, exist_ok=True)

print('Rutas creadas:')
for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    print(f'  {path}')

Rutas creadas:
  ./medallion/bronze/ventas
  ./medallion/silver/ventas
  ./medallion/gold


---
## 4. Capa Bronze — Ingesta de datos crudos

> **Bronze**: datos tal como llegan de la fuente. Sin limpieza. Solo se agrega metadata de ingesta.

In [5]:
# Leer el JSON crudo
df_raw = spark.read \
    .option('multiline', 'true') \
    .json(SOURCE_JSON)

print('=== Schema del dataset crudo ===')
df_raw.printSchema()

=== Schema del dataset crudo ===
root
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- status: string (nullable = true)
 |-- unit_price: double (nullable = true)



In [6]:
# Explorar los datos
print(f'Total registros: {df_raw.count()}')
df_raw.show(5, truncate=False)

Total registros: 30
+-----------+------------+--------+-------------------------+-----------+---------------+--------+----------+--------+----------+-------------------------+--------+---------+----------+
|category   |city        |country |customer_email           |customer_id|customer_name  |discount|order_date|order_id|product_id|product_name             |quantity|status   |unit_price|
+-----------+------------+--------+-------------------------+-----------+---------------+--------+----------+--------+----------+-------------------------+--------+---------+----------+
|Electrónica|Bogotá      |Colombia|ana.garcia@email.com     |C001       |Ana García     |0.1     |2024-01-05|ORD-001 |P101      |Laptop HP 15             |1       |entregado|1200.0    |
|Electrónica|Medellín    |Colombia|carlos.lopez@email.com   |C002       |Carlos López   |0.0     |2024-01-06|ORD-002 |P205      |Auriculares Sony WH      |2       |entregado|150.0     |
|Muebles    |Cali        |Colombia|maria.rodriguez

In [8]:
# Bronze: agregar metadata de ingesta y guardar en Parquet
from datetime import datetime

df_bronze = df_raw \
    .withColumn('ingestion_timestamp', F.current_timestamp()) \
    .withColumn('source_file', F.lit(SOURCE_JSON)) \
    .withColumn('ingestion_date', F.current_date())

# Escribir en formato Parquet (columnar, eficiente para analytics)
df_bronze.write \
    .mode('overwrite') \
    .partitionBy('ingestion_date') \
    .parquet(BRONZE_PATH)

print(f'Bronze guardado en: {BRONZE_PATH}')
print(f'Registros en Bronze: {df_bronze.count()}')

Bronze guardado en: ./medallion/bronze/ventas
Registros en Bronze: 30


In [9]:
# Verificar lo que se escribió en Bronze
df_bronze_check = spark.read.parquet(BRONZE_PATH)
df_bronze_check.select(
    'order_id', 'customer_name', 'product_name',
    'unit_price', 'status', 'ingestion_timestamp'
).show(5, truncate=False)

+--------+---------------+-------------------------+----------+---------+--------------------------+
|order_id|customer_name  |product_name             |unit_price|status   |ingestion_timestamp       |
+--------+---------------+-------------------------+----------+---------+--------------------------+
|ORD-001 |Ana García     |Laptop HP 15             |1200.0    |entregado|2026-04-03 22:03:29.638208|
|ORD-002 |Carlos López   |Auriculares Sony WH      |150.0     |entregado|2026-04-03 22:03:29.638208|
|ORD-003 |María Rodríguez|Silla Ergonómica         |350.0     |enviado  |2026-04-03 22:03:29.638208|
|ORD-004 |Ana García     |Mouse Inalámbrico        |45.0      |entregado|2026-04-03 22:03:29.638208|
|ORD-005 |Pedro Martínez |Libro Python Data Science|35.0      |entregado|2026-04-03 22:03:29.638208|
+--------+---------------+-------------------------+----------+---------+--------------------------+
only showing top 5 rows


---
## 5. Capa Silver — Limpieza y transformaciones

> **Silver**: datos limpios, validados y enriquecidos. Se corrigen tipos, se eliminan duplicados y se calculan columnas derivadas.

In [10]:
# Leer desde Bronze
df_b = spark.read.parquet(BRONZE_PATH)

# 1. Verificar nulos
print('=== Conteo de nulos por columna ===')
df_b.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'status']
]).show()

=== Conteo de nulos por columna ===
+--------+-----------+----------+--------+----------+------+
|order_id|customer_id|product_id|quantity|unit_price|status|
+--------+-----------+----------+--------+----------+------+
|       0|          0|         0|       0|         0|     0|
+--------+-----------+----------+--------+----------+------+



In [11]:
# 2. Verificar duplicados
total = df_b.count()
distinct = df_b.dropDuplicates(['order_id']).count()
print(f'Total registros: {total}')
print(f'Registros únicos por order_id: {distinct}')
print(f'Duplicados: {total - distinct}')

Total registros: 30
Registros únicos por order_id: 30
Duplicados: 0


In [12]:
# 3. Verificar valores de la columna status
df_b.groupBy('status').count().orderBy('count', ascending=False).show()

+---------+-----+
|   status|count|
+---------+-----+
|entregado|   23|
|  enviado|    4|
|cancelado|    3|
+---------+-----+



In [13]:
# Transformaciones Silver
df_silver = df_b \
    .dropDuplicates(['order_id']) \
    .filter(F.col('order_id').isNotNull()) \
    .filter(F.col('customer_id').isNotNull()) \
    .withColumn('order_date', F.to_date('order_date', 'yyyy-MM-dd')) \
    .withColumn('order_year',  F.year('order_date')) \
    .withColumn('order_month', F.month('order_date')) \
    .withColumn('unit_price',  F.col('unit_price').cast(DoubleType())) \
    .withColumn('quantity',    F.col('quantity').cast(IntegerType())) \
    .withColumn('discount',    F.col('discount').cast(DoubleType())) \
    .withColumn('gross_amount', F.round(F.col('quantity') * F.col('unit_price'), 2)) \
    .withColumn('net_amount',   F.round(
        F.col('quantity') * F.col('unit_price') * (1 - F.col('discount')), 2
    )) \
    .withColumn('is_cancelled', F.when(F.col('status') == 'cancelado', True).otherwise(False)) \
    .withColumn('status', F.lower(F.trim('status'))) \
    .drop('ingestion_date', 'source_file')

print(f'Silver: {df_silver.count()} registros')
df_silver.printSchema()

Silver: 30 registros
root
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- is_cancelled: boolean (nullable = false)



In [14]:
# Guardar Silver particionado por año y mes
df_silver.write \
    .mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(SILVER_PATH)

print(f'Silver guardado en: {SILVER_PATH}')

26/04/03 22:04:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver guardado en: ./medallion/silver/ventas


In [15]:
# Vista previa de Silver
spark.read.parquet(SILVER_PATH).select(
    'order_id', 'customer_name', 'product_name', 'category',
    'gross_amount', 'net_amount', 'status', 'order_date'
).show(8, truncate=False)

+--------+---------------+-------------------------+-----------+------------+----------+---------+----------+
|order_id|customer_name  |product_name             |category   |gross_amount|net_amount|status   |order_date|
+--------+---------------+-------------------------+-----------+------------+----------+---------+----------+
|ORD-013 |María Rodríguez|Teclado Mecánico         |Electrónica|120.0       |108.0     |entregado|2024-02-01|
|ORD-014 |Isabela Moreno |Escritorio de Pie        |Muebles    |500.0       |475.0     |enviado  |2024-02-03|
|ORD-015 |Felipe Vargas  |Auriculares Sony WH      |Electrónica|150.0       |150.0     |entregado|2024-02-05|
|ORD-016 |Natalia Ríos   |Laptop HP 15             |Electrónica|1200.0      |1020.0    |entregado|2024-02-07|
|ORD-017 |Pedro Martínez |Cámara Web HD            |Electrónica|80.0        |80.0      |cancelado|2024-02-08|
|ORD-018 |Camilo Ospina  |Monitor 24 pulgadas      |Electrónica|560.0       |504.0     |entregado|2024-02-10|
|ORD-019 |

---
## 6. Capa Gold — Agregaciones para análisis

> **Gold**: datos listos para consumo analítico. Se crean tablas de hechos y dimensiones, KPIs y reportes.

In [16]:
df_s = spark.read.parquet(SILVER_PATH)
df_active = df_s.filter(F.col('status') != 'cancelado')

# --- Gold 1: KPIs por mes ---
gold_monthly = df_active \
    .groupBy('order_year', 'order_month') \
    .agg(
        F.count('order_id').alias('total_orders'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.round(F.sum('gross_amount'), 2).alias('gross_revenue'),
        F.round(F.sum('net_amount'), 2).alias('net_revenue'),
        F.round(F.avg('net_amount'), 2).alias('avg_order_value')
    ) \
    .orderBy('order_year', 'order_month')

gold_monthly.write.mode('overwrite').parquet(f'{GOLD_PATH}/kpis_mensuales')
print('Gold: KPIs mensuales')
gold_monthly.show()

Gold: KPIs mensuales
+----------+-----------+------------+----------------+-------------+-----------+---------------+
|order_year|order_month|total_orders|unique_customers|gross_revenue|net_revenue|avg_order_value|
+----------+-----------+------------+----------------+-------------+-----------+---------------+
|      2024|          1|          11|               9|       3430.0|    3198.75|          290.8|
|      2024|          2|          11|              11|       4860.0|     4454.0|         404.91|
|      2024|          3|           5|               5|       1450.0|     1329.5|          265.9|
+----------+-----------+------------+----------------+-------------+-----------+---------------+



In [17]:
# --- Gold 2: Ventas por categoría de producto ---
gold_category = df_active \
    .groupBy('category') \
    .agg(
        F.count('order_id').alias('num_orders'),
        F.sum('quantity').alias('units_sold'),
        F.round(F.sum('net_amount'), 2).alias('net_revenue'),
        F.round(F.avg('unit_price'), 2).alias('avg_price')
    ) \
    .orderBy('net_revenue', ascending=False)

gold_category.write.mode('overwrite').parquet(f'{GOLD_PATH}/ventas_por_categoria')
print('Gold: Ventas por categoría')
gold_category.show()

Gold: Ventas por categoría
+-----------+----------+----------+-----------+---------+
|   category|num_orders|units_sold|net_revenue|avg_price|
+-----------+----------+----------+-----------+---------+
|Electrónica|        17|        25|     5920.5|   326.18|
|    Muebles|         6|         7|     2762.5|    425.0|
|     Libros|         4|        10|     299.25|     35.0|
+-----------+----------+----------+-----------+---------+



In [18]:
# --- Gold 3: Top clientes por revenue ---
gold_customers = df_active \
    .groupBy('customer_id', 'customer_name', 'city') \
    .agg(
        F.count('order_id').alias('num_orders'),
        F.round(F.sum('net_amount'), 2).alias('total_spent'),
        F.round(F.avg('net_amount'), 2).alias('avg_order')
    ) \
    .orderBy('total_spent', ascending=False)

gold_customers.write.mode('overwrite').parquet(f'{GOLD_PATH}/top_clientes')
print('Gold: Top clientes')
gold_customers.show(10)

Gold: Top clientes
+-----------+---------------+---------+----------+-----------+---------+
|customer_id|  customer_name|     city|num_orders|total_spent|avg_order|
+-----------+---------------+---------+----------+-----------+---------+
|       C001|     Ana García|   Bogotá|         3|     1405.0|   468.33|
|       C007|  Sofía Herrera| Medellín|         2|     1232.0|    616.0|
|       C013|   Natalia Ríos|   Bogotá|         1|     1020.0|   1020.0|
|       C008|   Diego Flores|     Cali|         2|      742.0|    371.0|
|       C006|    Juan Torres|   Bogotá|         2|      620.0|    310.0|
|       C010| Andrés Jiménez|   Bogotá|         2|      610.0|    305.0|
|       C002|   Carlos López| Medellín|         2|      580.0|    290.0|
|       C014|  Camilo Ospina|Manizales|         1|      504.0|    504.0|
|       C011| Isabela Moreno|   Bogotá|         1|      475.0|    475.0|
|       C003|María Rodríguez|     Cali|         2|      440.5|   220.25|
+-----------+---------------+---

In [19]:
# --- Gold 4: Top productos ---
gold_products = df_active \
    .groupBy('product_id', 'product_name', 'category') \
    .agg(
        F.sum('quantity').alias('units_sold'),
        F.round(F.sum('net_amount'), 2).alias('net_revenue')
    ) \
    .orderBy('net_revenue', ascending=False)

gold_products.write.mode('overwrite').parquet(f'{GOLD_PATH}/top_productos')
print('Gold: Top productos')
gold_products.show()

Gold: Top productos
+----------+--------------------+-----------+----------+-----------+
|product_id|        product_name|   category|units_sold|net_revenue|
+----------+--------------------+-----------+----------+-----------+
|      P101|        Laptop HP 15|Electrónica|         3|     3180.0|
|      P901|   Escritorio de Pie|    Muebles|         3|     1450.0|
|      P310|    Silla Ergonómica|    Muebles|         4|     1312.5|
|      P801| Monitor 24 pulgadas|Electrónica|         4|     1064.0|
|      P205| Auriculares Sony WH|Electrónica|         6|      832.5|
|      P602|    Teclado Mecánico|Electrónica|         3|      342.0|
|      P501|Libro Python Data...|     Libros|        10|     299.25|
|      P410|   Mouse Inalámbrico|Electrónica|         6|      270.0|
|      P710|       Cámara Web HD|Electrónica|         3|      232.0|
+----------+--------------------+-----------+----------+-----------+



---
## 7. Consultas analíticas sobre Gold

In [20]:
# Registrar Silver como vista temporal para usar Spark SQL
df_s.createOrReplaceTempView('ventas')

# Spark SQL — exactamente igual a SQL estándar
spark.sql('''
SELECT
    category,
    COUNT(order_id)       AS num_ordenes,
    SUM(quantity)         AS unidades,
    ROUND(SUM(net_amount),2) AS ingresos_netos
FROM ventas
WHERE status != 'cancelado'
GROUP BY category
ORDER BY ingresos_netos DESC
''').show()

+-----------+-----------+--------+--------------+
|   category|num_ordenes|unidades|ingresos_netos|
+-----------+-----------+--------+--------------+
|Electrónica|         17|      25|        5920.5|
|    Muebles|          6|       7|        2762.5|
|     Libros|          4|      10|        299.25|
+-----------+-----------+--------+--------------+



In [21]:
# Tasa de cancelación
spark.sql('''
SELECT
    CONCAT(order_year, '-', LPAD(order_month, 2, '0')) AS mes,
    COUNT(*) AS total,
    SUM(CASE WHEN status = 'cancelado' THEN 1 ELSE 0 END) AS cancelados,
    ROUND(100.0 * SUM(CASE WHEN status = 'cancelado' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_cancelacion
FROM ventas
GROUP BY mes
ORDER BY mes
''').show()

+-------+-----+----------+---------------+
|    mes|total|cancelados|pct_cancelacion|
+-------+-----+----------+---------------+
|2024-01|   12|         1|            8.3|
|2024-02|   12|         1|            8.3|
|2024-03|    6|         1|           16.7|
+-------+-----+----------+---------------+



In [22]:
# Window function — ranking de clientes por revenue dentro de cada ciudad
from pyspark.sql.window import Window

window_city = Window.partitionBy('city').orderBy(F.desc('total_spent'))

spark.read.parquet(f'{GOLD_PATH}/top_clientes') \
    .withColumn('rank_in_city', F.rank().over(window_city)) \
    .filter(F.col('rank_in_city') == 1) \
    .select('city', 'customer_name', 'total_spent', 'num_orders') \
    .orderBy('total_spent', ascending=False) \
    .show()

+------------+---------------+-----------+----------+
|        city|  customer_name|total_spent|num_orders|
+------------+---------------+-----------+----------+
|      Bogotá|     Ana García|     1405.0|         3|
|    Medellín|  Sofía Herrera|     1232.0|         2|
|        Cali|   Diego Flores|      742.0|         2|
|   Manizales|  Camilo Ospina|      504.0|         1|
|      Cúcuta|  Mauricio Peña|      382.5|         1|
|   Cartagena|  Laura Sánchez|      350.0|         1|
|     Pereira|  Felipe Vargas|      150.0|         1|
| Santa Marta|Sebastián Muñoz|       90.0|         1|
|Barranquilla| Pedro Martínez|      89.25|         1|
| Bucaramanga| Valentina Cruz|       35.0|         1|
+------------+---------------+-----------+----------+



In [23]:
# Cerrar SparkSession
spark.stop()
print('SparkSession cerrada.')

SparkSession cerrada.


---
## 8. Instructivo: Migrar la Arquitectura Medallion a AWS S3 + Glue

### Resumen de la arquitectura en AWS

```
Fuente de datos
     │
     ▼
S3: s3://mi-bucket/raw/ventas/       ← Bronze (JSON crudo)
     │
AWS Glue Job (PySpark)
     │
     ▼
S3: s3://mi-bucket/silver/ventas/    ← Silver (Parquet limpio)
     │
AWS Glue Job (PySpark)
     │
     ▼
S3: s3://mi-bucket/gold/             ← Gold (Parquet agregado)
     │
AWS Glue Data Catalog  ←──────────── Crawlers actualizan el catálogo
     │
Amazon Athena (SQL sobre S3)
```

---

### Paso 1 — Crear la estructura de buckets en S3

```bash
# Con AWS CLI
BUCKET=mi-bucket-medallion-lab-acosta-samuel

aws s3 mb s3://$BUCKET
aws s3api put-object --bucket $BUCKET --key bronze/ventas/
aws s3api put-object --bucket $BUCKET --key silver/ventas/
aws s3api put-object --bucket $BUCKET --key gold/kpis_mensuales/
aws s3api put-object --bucket $BUCKET --key gold/ventas_por_categoria/
aws s3api put-object --bucket $BUCKET --key gold/top_clientes/
aws s3api put-object --bucket $BUCKET --key gold/top_productos/
aws s3api put-object --bucket $BUCKET --key scripts/  # Para scripts Glue

# Subir el dataset JSON a Bronze
aws s3 cp data/ventas_ecommerce.json s3://$BUCKET/bronze/ventas/ventas_ecommerce.json
```

---

### Paso 2 — Crear IAM Role para AWS Glue

1. Ve a **IAM** > **Roles** > **Create role**.
2. Trusted entity: **AWS Service** > **Glue**.
3. Adjunta las políticas:
   - `AWSGlueServiceRole`
   - `AmazonS3FullAccess` (o una política más restrictiva solo para tu bucket).
4. Nombre del rol: `GlueLabRole`.

---

### Paso 3 — Script de Glue para Bronze → Silver

Guarda el siguiente script como `glue_bronze_to_silver.py` y súbelo a `s3://$BUCKET/scripts/`:

```python
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql.types import *

args = getResolvedOptions(sys.argv, ['JOB_NAME', 'BUCKET'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

BUCKET = args['BUCKET']  # ej: mi-bucket-medallion-lab
BRONZE = f's3://{BUCKET}/bronze/ventas/'
SILVER = f's3://{BUCKET}/silver/ventas/'

# Leer Bronze
df_b = spark.read.option('multiline', 'true').json(BRONZE)

# Transformaciones Silver (iguales al notebook local)
df_silver = df_b \
    .dropDuplicates(['order_id']) \
    .filter(F.col('order_id').isNotNull()) \
    .withColumn('order_date',   F.to_date('order_date', 'yyyy-MM-dd')) \
    .withColumn('order_year',   F.year('order_date')) \
    .withColumn('order_month',  F.month('order_date')) \
    .withColumn('unit_price',   F.col('unit_price').cast(DoubleType())) \
    .withColumn('quantity',     F.col('quantity').cast(IntegerType())) \
    .withColumn('discount',     F.col('discount').cast(DoubleType())) \
    .withColumn('gross_amount', F.round(F.col('quantity') * F.col('unit_price'), 2)) \
    .withColumn('net_amount',   F.round(
        F.col('quantity') * F.col('unit_price') * (1 - F.col('discount')), 2)) \
    .withColumn('status',       F.lower(F.trim('status')))

# Escribir Silver en S3
df_silver.write \
    .mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(SILVER)

job.commit()
```

```bash
# Subir el script
aws s3 cp glue_bronze_to_silver.py s3://$BUCKET/scripts/
```

---

### Paso 4 — Crear el Glue Job desde la consola

1. Ve a **AWS Glue** > **ETL Jobs** > **Create job**.
2. Selecciona **Script editor** > **Upload an existing script**.
3. Sube `glue_bronze_to_silver.py`.
4. Configuración:
   - **Name**: `bronze-to-silver-ventas`
   - **IAM Role**: `GlueLabRole`
   - **Glue version**: Glue 4.0 (Spark 3.3)
   - **Worker type**: G.1X (el más pequeño del Free Tier)
   - **Number of workers**: 2
5. En **Job parameters** agrega:
   - Key: `--BUCKET` | Value: `mi-bucket-medallion-lab`
6. Haz clic en **Save** y luego **Run**.

---

### Paso 5 — Crear Crawlers para el Glue Data Catalog

1. Ve a **AWS Glue** > **Crawlers** > **Create crawler**.
2. Name: `crawler-silver-ventas`
3. Data source: S3 — `s3://mi-bucket-medallion-lab/silver/ventas/`
4. IAM Role: `GlueLabRole`
5. Target database: crea una nueva llamada `medallion_db`
6. Haz clic en **Create** y luego **Run crawler**.
7. Repite para `gold/` apuntando a cada subcarpeta.

---

### Paso 6 — Consultar con Amazon Athena

1. Ve a **Amazon Athena** > **Query editor**.
2. Configura un bucket para resultados: `s3://mi-bucket-medallion-lab/athena-results/`
3. Selecciona la base de datos `medallion_db`.
4. Ejecuta consultas SQL directamente sobre los datos en S3:

```sql
-- Ver tablas disponibles
SHOW TABLES IN medallion_db;

-- Consulta sobre Silver
SELECT
    category,
    COUNT(*) AS num_ordenes,
    ROUND(SUM(net_amount), 2) AS ingresos_netos
FROM medallion_db.ventas
WHERE status != 'cancelado'
GROUP BY category
ORDER BY ingresos_netos DESC;

-- Athena usa particiones automáticamente (más barato y rápido)
SELECT *
FROM medallion_db.ventas
WHERE order_year = 2024 AND order_month = 1;
```

---

### Costos estimados (Free Tier)

| Servicio | Free Tier |
|---|---|
| S3 | 5 GB almacenamiento, 20K GET, 2K PUT |
| AWS Glue | 1 millón de objetos en Data Catalog, DPU-horas no incluidas* |
| Amazon Athena | 5 TB de datos escaneados por mes |

> *Glue ETL Jobs no están incluidos en el Free Tier pero tienen costo muy bajo (~$0.44/DPU-hora). Un job pequeño cuesta centavos de dólar.

---

### Buenas prácticas para producción
- Usa **particiones** en S3 por fecha para reducir el escaneo en Athena.
- Prefiere **Parquet** o **ORC** sobre CSV/JSON en Silver y Gold.
- Usa **Glue Bookmarks** para procesar solo datos nuevos (incremental).
- Implementa **data quality checks** en Silver con AWS Deequ o Great Expectations.
- Usa **Glue Workflows** para orquestar Bronze → Silver → Gold automáticamente.